# Calibration analysis — Logistic Regression and SVGP

This notebook **does not implement coverage**.

For binary urban-expansion probabilities, calibration means that predictions
near a probability level should match the observed event frequency at that
level. Example: among cells predicted near 0.8, roughly 80% should convert in
a well-calibrated model.

The notebook reads existing OOF predictions only; it does not train models.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.models.evaluation.calibration import calibration_metrics, reliability_table

LR_PATH = Path("reports/modeling/logistic_regression/predictions/logistic_oof_predictions.parquet")
SVGP_PATH = Path("reports/modeling/svgp/predictions/svgp_oof_predictions.parquet")

OUT = Path("reports/modeling/calibration")
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
def resolve_columns(frame):
    target_candidates = [
        "target_transition_5y",
        "target",
        "y_true",
    ]
    probability_candidates = [
        "probability_raw",
        "predicted_probability",
        "probability",
    ]

    target = next((c for c in target_candidates if c in frame.columns), None)
    probability = next((c for c in probability_candidates if c in frame.columns), None)

    if target is None or probability is None:
        raise ValueError(
            f"Could not resolve target/probability columns. Columns={list(frame.columns)}"
        )
    return target, probability


models = {
    "Logistic Regression": pd.read_parquet(LR_PATH),
    "SVGP": pd.read_parquet(SVGP_PATH),
}


In [ ]:
summary_rows = []
reliability_rows = []

for model_name, frame in models.items():
    target, probability = resolve_columns(frame)

    if "fold" in frame.columns:
        groups = frame.groupby("fold")
    else:
        groups = [("all", frame)]

    for fold, part in groups:
        metrics = calibration_metrics(
            part[target].to_numpy(),
            part[probability].to_numpy(),
            n_bins=10,
            strategy="quantile",
        )
        summary_rows.append(
            {
                "model": model_name,
                "fold": fold,
                **metrics,
            }
        )

        reliability = reliability_table(
            part[target].to_numpy(),
            part[probability].to_numpy(),
            n_bins=10,
            strategy="quantile",
        )
        reliability["model"] = model_name
        reliability["fold"] = fold
        reliability_rows.append(reliability)

summary = pd.DataFrame(summary_rows)
reliability = pd.concat(reliability_rows, ignore_index=True)

summary.to_csv(OUT / "calibration_summary.csv", index=False)
reliability.to_csv(OUT / "reliability_bins.csv", index=False)

summary


In [ ]:
for model_name in reliability["model"].unique():
    part = reliability[reliability["model"] == model_name]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], linestyle="--")

    for fold, fold_part in part.groupby("fold"):
        ax.plot(
            fold_part["mean_predicted_probability"],
            fold_part["observed_conversion_rate"],
            marker="o",
            label=f"Fold {fold}",
        )

    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed conversion rate")
    ax.set_title(f"Reliability — {model_name}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        OUT / f"reliability_{model_name.lower().replace(' ', '_')}.png",
        dpi=180,
    )
    plt.show()


## Interpretation guide

- **Brier / Log Loss**: overall probability quality.
- **ECE**: average reliability gap; lower is better.
- **Calibration intercept**: ideal ≈ 0.
- **Calibration slope**: ideal ≈ 1.
- **Probability bias**: mean predicted probability minus observed prevalence.
- **Temporal robustness**: inspect the metrics fold by fold, not only their mean.

Coverage/80% intervals are intentionally deferred until the intended definition
is clarified with Jess.